# 🔌 PCB Router — Colab Training Notebook

**Architecture:** ViT-Small + Hetero-GAT + Cross-Attention Fusion + Spatial JEPA + PPO  
**Environment:** Multi-layer PCB routing with A* pathfinding  
**Training:** Proximal Policy Optimization with JEPA world-model auxiliary loss

---
### Quick Start
1. **Edit `CONFIG`** in Cell 2  
2. **Run Cell 3** — mount Google Drive (recommended)  
3. **Run Cell 4** — upload code OR clone from GitHub, then install deps  
4. **Run Cell 5** — initialize trainer  
5. **Run Cell 6** — train with live visuals  

> 💡 **Enable GPU:** Runtime → Change runtime type → T4 GPU (20–50× faster than CPU)


In [ ]:
# ================================================================
#  CELL 2 — CONFIGURATION  (Edit this cell before running)
# ================================================================

CONFIG = {
    # -----------------------------------------------------------
    #  How to get the code into Colab — pick ONE method:
    #
    #  Method A (easiest): Upload a zip of the project.
    #    Set REPO_URL = None  →  Cell 4 will prompt you to upload a zip.
    #
    #  Method B: Clone from GitHub.
    #    Set REPO_URL = "https://github.com/YOUR_USERNAME/Router.git"
    # -----------------------------------------------------------
    "REPO_URL": None,   # <-- None = zip upload, or paste your GitHub URL here
    "REPO_DIR": "/content/Router",

    # -----------------------------------------------------------
    #  Checkpoints
    # -----------------------------------------------------------
    # Google Drive path (survives session restarts — recommended):
    "CHECKPOINT_DIR": "/content/drive/MyDrive/pcb_router/checkpoints",
    # Local Colab path (lost when session ends):
    # "CHECKPOINT_DIR": "/content/checkpoints",

    # Resume from a previous checkpoint, or None to start fresh:
    "LOAD_CHECKPOINT": None,
    # Example: "/content/drive/MyDrive/pcb_router/checkpoints/checkpoint_50000.pt"

    # -----------------------------------------------------------
    #  Training
    # -----------------------------------------------------------
    "TOTAL_TIMESTEPS": 5_000_000,
    "SAVE_INTERVAL":   10_000,     # Save a .pt checkpoint every N timesteps
    "VIZ_INTERVAL":    1,          # Show live dashboard every N rollout updates

    # -----------------------------------------------------------
    #  DreamerV3 Hyperparameters  (override configs/training.yaml)
    # -----------------------------------------------------------
    "REAL_STEPS_PER_ITERATION":  64,
    "TRAIN_RATIO":               100,
    "REPLAY_BUFFER_SIZE":        5000,
    "IMAGINE_BATCH_SIZE":        512,
    "IMAGINATION_HORIZON_START": 5,
    "IMAGINATION_HORIZON_END":   15,
    "WORLD_MODEL_LR":            3e-4,
    "ACTOR_LR":                  8e-5,
    "CRITIC_LR":                 8e-5,
    "GAMMA":                     0.997,
    "LAMBDA":                    0.95,

    # -----------------------------------------------------------
    #  Model  (override configs/model.yaml)
    # -----------------------------------------------------------
    "MAX_GRID_SIZE": 256,          # Do NOT exceed 512 without 16GB+ VRAM

    # -----------------------------------------------------------
    #  Logging
    # -----------------------------------------------------------
    "USE_WANDB":      False,
    "WANDB_PROJECT":  "pcb-router",
    "WANDB_RUN_NAME": None,
    "COMPILE_MODELS": True,
}

print("Config loaded:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


In [ ]:
# ================================================================
#  CELL 3 — MOUNT GOOGLE DRIVE
#  Skip if you set CHECKPOINT_DIR to a local /content/ path
# ================================================================
import os

USE_DRIVE = CONFIG["CHECKPOINT_DIR"].startswith("/content/drive")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(CONFIG["CHECKPOINT_DIR"], exist_ok=True)
    print(f"Drive mounted. Checkpoints: {CONFIG['CHECKPOINT_DIR']}")
else:
    os.makedirs(CONFIG["CHECKPOINT_DIR"], exist_ok=True)
    print(f"Using local storage: {CONFIG['CHECKPOINT_DIR']}")
    print("WARNING: Local Colab storage is lost when the session ends!")


In [ ]:
# ================================================================
#  CELL 4 — GET CODE + INSTALL DEPENDENCIES
#
#  Supports two methods (set by REPO_URL in Cell 2):
#    • REPO_URL = None        → prompts you to upload a .zip of the project
#    • REPO_URL = "https://…" → clones from GitHub
# ================================================================
import subprocess, sys, os, zipfile, shutil

REPO_DIR = CONFIG["REPO_DIR"]
REPO_URL = CONFIG["REPO_URL"]

# ── Method A: Zip upload ──────────────────────────────────────────────────────
if REPO_URL is None:
    if not os.path.exists(REPO_DIR):
        print("No REPO_URL set — please upload your project zip file.")
        print("Steps:")
        print("  1. On your PC: right-click the 'Router' folder → Send to → Compressed (zipped) folder")
        print("  2. A file browser will appear below — select your Router.zip")
        from google.colab import files
        uploaded = files.upload()          # opens file picker
        zip_name = list(uploaded.keys())[0]
        print(f"\nUploaded: {zip_name}  ({len(uploaded[zip_name]):,} bytes)")

        # Create temporary extraction folder
        temp_dir = "/content/temp_extract"
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)
        os.makedirs(temp_dir)

        print(f"Extracting {zip_name}...")
        with zipfile.ZipFile(zip_name, "r") as zf:
            zf.extractall(temp_dir)

        # Robustly locate the folder containing the project files (pcb_router folder or setup.py)
        target_src_dir = None
        for root, dirs, files_in_dir in os.walk(temp_dir):
            if "pcb_router" in dirs or "setup.py" in files_in_dir:
                target_src_dir = root
                break

        if target_src_dir is None:
            shutil.rmtree(temp_dir, ignore_errors=True)
            os.remove(zip_name)
            raise FileNotFoundError(
                "Could not find 'pcb_router' folder or 'setup.py' in the uploaded zip file.\n"
                "Please make sure you zipped the correct project folder."
            )

        # Move the found project directory to /content/Router
        if os.path.exists(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        shutil.move(target_src_dir, REPO_DIR)

        # Clean up temp files
        shutil.rmtree(temp_dir, ignore_errors=True)
        try:
            os.remove(zip_name)
        except:
            pass

        print(f"Successfully extracted and moved project files to {REPO_DIR}")
    else:
        print(f"Repo already at {REPO_DIR} — skipping upload.")

# ── Method B: GitHub clone ────────────────────────────────────────────────────
else:
    PLACEHOLDER = "YOUR_USERNAME"
    if PLACEHOLDER in REPO_URL:
        raise ValueError(
            f"REPO_URL still contains placeholder '{PLACEHOLDER}'.\n"
            "Either set REPO_URL = None to use zip upload,\n"
            "or replace REPO_URL with your actual GitHub URL in Cell 2."
        )

    if not os.path.exists(REPO_DIR):
        print(f"Cloning {REPO_URL} ...")
        r = subprocess.run(["git", "clone", REPO_URL, REPO_DIR],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(
                f"git clone failed (exit {r.returncode}):\n{r.stderr}\n\n"
                "Tip: Make sure your repo is public, or use REPO_URL = None to upload a zip instead."
            )
        print(r.stdout or "Cloned successfully.")
    else:
        print(f"Repo found at {REPO_DIR} — pulling latest...")
        r = subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True, text=True)
        print(r.stdout or r.stderr)

# ── Verify the directory structure is correct ───────────────────────────
if not os.path.exists(REPO_DIR):
    raise FileNotFoundError(f"Project directory not found at {REPO_DIR}.")

# Ensure path is clean and active in Python paths
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Force-remove any outdated/corrupted setuptools egg links or build caches
for folder in ["build", "pcb_router.egg-info"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)

print(f"\nWorking directory set to: {os.getcwd()}")
print(f"Directory Contents: {os.listdir('.')}")

# ── Install deps ──────────────────────────────────────────────────────────────
import torch
TORCH_VER = torch.__version__.split("+")[0]
HAS_CUDA  = torch.cuda.is_available()
CUDA_TAG  = "cu" + torch.version.cuda.replace(".", "") if HAS_CUDA else "cpu"
PYG_URL   = f"https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html"

print(f"\ntorch {TORCH_VER} | CUDA: {HAS_CUDA} ({CUDA_TAG})")
if HAS_CUDA:
    gb = torch.cuda.get_device_properties(0).total_memory // 1024**3
    print(f"GPU: {torch.cuda.get_device_name(0)} ({gb}GB)")

print("Installing dependencies (this takes ~2 min the first time)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch-scatter", "torch-sparse", "-f", PYG_URL], check=True)
# Install in editable mode to register package path properly
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "gymnasium", "wandb", "tqdm", "dataclasses-json"], check=True)

import torch_geometric, torch_scatter
print(f"\ntorch-geometric: {torch_geometric.__version__}")
print("torch-scatter:  OK")
print("\nAll dependencies installed! ✓")


In [ ]:
# ================================================================
#  CELL 5 — INITIALIZE TRAINER
# ================================================================
import os, sys

# ── Clear Python import cache to force reloading from disk ─────
for key in list(sys.modules.keys()):
    if key == "pcb_router" or key.startswith("pcb_router."):
        del sys.modules[key]

sys.path.insert(0, CONFIG["REPO_DIR"])
os.chdir(CONFIG["REPO_DIR"])

from pcb_router.training.trainer import PPOJEPATrainer, DreamerJEPATrainer

# Initialize the new DreamerJEPATrainer by default
trainer = DreamerJEPATrainer(
    config_path="configs/training.yaml",
    model_config_path="configs/model.yaml",
    curriculum_config_path="configs/curriculum.yaml",
    device="auto",                             # auto-selects GPU if available
    checkpoint_dir=CONFIG["CHECKPOINT_DIR"],
    load_checkpoint_path=CONFIG["LOAD_CHECKPOINT"],
)

# Apply CONFIG overrides dynamically depending on trainer type
if isinstance(trainer, DreamerJEPATrainer):
    trainer.real_steps_per_iteration = CONFIG.get("REAL_STEPS_PER_ITERATION", 64)
    trainer.train_ratio = CONFIG.get("TRAIN_RATIO", 100)
    trainer.replay_buffer.capacity_episodes = CONFIG.get("REPLAY_BUFFER_SIZE", 5000)
    trainer.imagine_batch_size = CONFIG.get("IMAGINE_BATCH_SIZE", 512)
    trainer.imagination_horizon_start = CONFIG.get("IMAGINATION_HORIZON_START", 5)
    trainer.imagination_horizon_end = CONFIG.get("IMAGINATION_HORIZON_END", 15)
    trainer.gamma = CONFIG.get("GAMMA", 0.997)
    trainer.lambda_ = CONFIG.get("LAMBDA", 0.95)
    trainer.compile_models = CONFIG.get("COMPILE_MODELS", True)
    
    for pg in trainer.wm_opt.param_groups:
        pg["lr"] = CONFIG.get("WORLD_MODEL_LR", 3e-4)
    for pg in trainer.actor_opt.param_groups:
        pg["lr"] = CONFIG.get("ACTOR_LR", 8e-5)
    for pg in trainer.critic_opt.param_groups:
        pg["lr"] = CONFIG.get("CRITIC_LR", 8e-5)
else:
    trainer.train_cfg["ppo"]["batch_size"]         = CONFIG.get("BATCH_SIZE", 16)
    trainer.train_cfg["ppo"]["num_rollout_steps"]  = CONFIG.get("NUM_ROLLOUT_STEPS", 64)
    trainer.train_cfg["ppo"]["num_epochs"]         = CONFIG.get("NUM_EPOCHS", 4)
    trainer.train_cfg["ppo"]["learning_rate"]      = CONFIG.get("LEARNING_RATE", 3e-4)
    trainer.train_cfg["ppo"]["gamma"]              = CONFIG.get("GAMMA", 0.99)
    trainer.train_cfg["ppo"]["gae_lambda"]         = CONFIG.get("GAE_LAMBDA", 0.95)
    for pg in trainer.optimizer.param_groups:
        pg["lr"] = CONFIG.get("LEARNING_RATE", 3e-4)

trainer.train_cfg["training"]["save_interval"] = CONFIG["SAVE_INTERVAL"]

if CONFIG["USE_WANDB"]:
    import wandb
    wandb.init(
        project=CONFIG["WANDB_PROJECT"],
        name=CONFIG["WANDB_RUN_NAME"],
        config=CONFIG
    )
    print(f"W&B run: {wandb.run.url}")

print("Trainer ready!")
print(f"  Device:           {trainer.device}")
print(f"  Checkpoint dir:   {trainer.checkpoint_dir}")
print(f"  Resuming from:    {CONFIG['LOAD_CHECKPOINT'] or 'scratch'}")
print(f"  Timesteps so far: {trainer.total_timesteps:,}")


In [ ]:
# ================================================================
#  CELL 6 — TRAINING WITH LIVE VISUALS
#
#  After each rollout update, displays:
#    • Training dashboard: completion rate + 3 loss curves
#    • Current board routing state
#
#  Interrupt (Ctrl+C / Runtime > Interrupt) → emergency checkpoint saved.
# ================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import IPython.display as ipydisplay
from IPython.display import clear_output
import numpy as np, os

from pcb_router.visualization.renderer import BoardRenderer
renderer = BoardRenderer(theme_dark=True)

_update_count = [0]

BG      = "#0d0e1a"
PANEL   = "#13142b"
BORDER  = "#2a2d50"
WHITE   = "#e8eaf6"
TEAL    = "#10B981"
BLUE    = "#3B82F6"
PURPLE  = "#8B5CF6"
AMBER   = "#F59E0B"


def on_update(trainer, metrics):
    _update_count[0] += 1
    if _update_count[0] % CONFIG["VIZ_INTERVAL"] != 0:
        return

    if CONFIG["USE_WANDB"]:
        import wandb
        wandb.log(metrics)

    h   = trainer.metrics_history
    ts  = metrics["timesteps"]
    stg = h["stage"][-1] if h["stage"] else "-"

    # ── Dashboard figure ─────────────────────────────────────────
    fig = plt.figure(figsize=(16, 9), facecolor=BG)
    gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35,
                            left=0.06, right=0.97, top=0.88, bottom=0.09)

    fig.suptitle(
        f"PCB Router Training   |   Step {ts:,}   |   Stage: {stg}",
        color=WHITE, fontsize=15, fontweight="bold", y=0.96
    )

    is_dreamer = "loss_wm" in h
    if is_dreamer:
        metrics_to_plot = [
            (h["completion_rate"], "Routing Completion Rate", TEAL,   "Rate",  (0, 0)),
            (h["loss_actor"],      "Actor Loss (Dreamer)",    BLUE,   "Loss",  (0, 1)),
            (h["loss_wm"],         "JEPA World Model Loss",   PURPLE, "Loss",  (1, 0)),
            (h["loss_critic"],     "Critic Loss (Dreamer)",   AMBER,  "Loss",  (1, 1)),
        ]
    else:
        metrics_to_plot = [
            (h["completion_rate"], "Routing Completion Rate", TEAL,   "Rate",  (0, 0)),
            (h["loss_policy"],     "Policy Loss (PPO)",       BLUE,   "Loss",  (0, 1)),
            (h["loss_jepa"],       "JEPA World Model Loss",   PURPLE, "Loss",  (1, 0)),
            (h["loss_value"],      "Value Function Loss",     AMBER,  "Loss",  (1, 1)),
        ]

    for y_data, title, color, ylabel, (row, col) in metrics_to_plot:
        ax = fig.add_subplot(gs[row, col])
        ax.set_facecolor(PANEL)
        for spine in ax.spines.values():
            spine.set_color(BORDER)
        ax.tick_params(colors="#888", labelsize=8)
        ax.set_xlabel("Updates", color="#888", fontsize=8)
        ax.set_ylabel(ylabel, color="#888", fontsize=8)
        ax.set_title(title, color=WHITE, fontsize=10, pad=6)
        ax.grid(True, color=BORDER, linestyle="--", linewidth=0.4, alpha=0.7)

        if len(y_data) > 0:
            xs = np.arange(len(y_data))
            ax.plot(xs, y_data, color=color, linewidth=0.8, alpha=0.25)
            # Contiguous moving average starting from step 0
            ma = [np.mean(y_data[max(0, i - 19) : i + 1]) for i in range(len(y_data))]
            ax.plot(xs, ma, color=color, linewidth=2.0)
            ax.annotate(
                f"{y_data[-1]:.4f}",
                xy=(xs[-1], y_data[-1]),
                color=color, fontsize=8, ha="right", va="bottom",
                bbox=dict(boxstyle="round,pad=0.25", fc=BG, ec=color, lw=0.8)
            )

    # ── Board state panel (split by active layers & AI heatmaps) ──
    try:
        state = trainer.env.board_state
        board = trainer.env.board
        num_layers = board.num_layers
        import matplotlib.patches as mpatches

        if num_layers == 2:
            sub_gs = gridspec.GridSpecFromSubplotSpec(
                2, 2, subplot_spec=gs[:, 2:], hspace=0.4, wspace=0.3
            )
            layer_names = ["Top Layer (L0)", "Bottom Layer (L1)"]
            layer_colors_title = ["#F43F5E", "#06B6D4"]
            
            # Row 0: Traces (A* drawn)
            for l in range(2):
                ax_lay = fig.add_subplot(sub_gs[0, l])
                ax_lay.set_facecolor(PANEL)
                
                t_color = layer_colors_title[l]
                active_traces = [s for s in state.traces if s.layer == l]
                ax_lay.set_title(
                    f"{layer_names[l]} - Traces (A*)\n({len(active_traces)} routed)",
                    color=t_color, fontsize=9, pad=4
                )
                ax_lay.set_xticks([]); ax_lay.set_yticks([])
                for spine in ax_lay.spines.values():
                    spine.set_color(BORDER)

                # Draw obstacles
                for obs in board.obstacles:
                    if obs.layer == -1 or obs.layer == l:
                        ax_lay.add_patch(mpatches.Rectangle(
                            (obs.x, obs.y), obs.width, obs.height,
                            fc="#EF4444", alpha=0.15, lw=0))

                # Draw components
                for comp in board.components:
                    ax_lay.add_patch(mpatches.Rectangle(
                        (comp.x, comp.y), comp.width, comp.height,
                        fc="#1e2040", ec="#4B5563", lw=1.0, alpha=0.7))

                # Draw traces
                net_colors = ["#3B82F6","#10B981","#EC4899","#8B5CF6","#06B6D4",
                              "#F59E0B","#14B8A6","#6366F1","#A855F7","#10B981"]
                for seg in state.traces:
                    if seg.layer == l:
                        c = net_colors[seg.net_id % len(net_colors)]
                        ax_lay.plot([seg.start_x, seg.end_x], [seg.start_y, seg.end_y],
                                    color=c, linewidth=2.0, solid_capstyle="round")

                # Draw vias
                for via in state.vias:
                    ax_lay.add_patch(mpatches.Circle(
                        (via.x, via.y), radius=3.5,
                        fc="#EAB308", ec=WHITE, lw=0.8, alpha=0.9, zorder=5))
                    ax_lay.add_patch(mpatches.Circle(
                        (via.x, via.y), radius=1.2,
                        fc=BG, zorder=6))

                # Draw pins
                for pin in board.pins.values():
                    c = net_colors[pin.net_id % len(net_colors)]
                    is_active = (pin.layer == l)
                    alpha = 0.95 if is_active else 0.25
                    ls = 'solid' if is_active else 'dashed'
                    ax_lay.add_patch(mpatches.Circle(
                        (pin.global_x, pin.global_y), radius=2.5,
                        fc=c, ec=WHITE if is_active else "#555566", lw=0.6, alpha=alpha, linestyle=ls))

                ax_lay.set_xlim(0, board.width)
                ax_lay.set_ylim(0, board.height)
                ax_lay.set_aspect("equal")

            # Row 1: AI Heatmaps (AI drawn)
            for l in range(2):
                ax_hm = fig.add_subplot(sub_gs[1, l])
                ax_hm.set_facecolor(PANEL)
                
                t_color = layer_colors_title[l]
                ax_hm.set_title(
                    f"{layer_names[l]} - Heatmap (AI)",
                    color=t_color, fontsize=9, pad=4
                )
                ax_hm.set_xticks([]); ax_hm.set_yticks([])
                for spine in ax_hm.spines.values():
                    spine.set_color(BORDER)

                # Show heatmap grid if available
                if hasattr(trainer, 'last_heatmap') and trainer.last_heatmap is not None:
                    ax_hm.imshow(
                        trainer.last_heatmap[l],
                        cmap='magma', origin='lower',
                        extent=(0, board.width, 0, board.height),
                        alpha=0.85
                    )
                else:
                    ax_hm.text(0.5, 0.5, "No Heatmap Yet", color="#888",
                               transform=ax_hm.transAxes, ha="center", va="center")

                # Draw pins on top of heatmap for reference
                for pin in board.pins.values():
                    net_colors = ["#3B82F6","#10B981","#EC4899","#8B5CF6","#06B6D4",
                                  "#F59E0B","#14B8A6","#6366F1","#A855F7","#10B981"]
                    c = net_colors[pin.net_id % len(net_colors)]
                    is_active = (pin.layer == l)
                    if is_active:
                        ax_hm.add_patch(mpatches.Circle(
                            (pin.global_x, pin.global_y), radius=2.5,
                            fc=c, ec=WHITE, lw=0.6, alpha=0.9))

                ax_hm.set_xlim(0, board.width)
                ax_hm.set_ylim(0, board.height)
                ax_hm.set_aspect("equal")
        else:
            rows, cols = int(np.ceil(num_layers / 2)), 2
            sub_gs = gridspec.GridSpecFromSubplotSpec(
                rows, cols, subplot_spec=gs[:, 2:], hspace=0.3, wspace=0.25
            )
            for l in range(num_layers):
                r_idx = l // cols
                c_idx = l % cols
                ax_lay = fig.add_subplot(sub_gs[r_idx, c_idx])
                ax_lay.set_facecolor(PANEL)
                ax_lay.set_title(f"Layer {l}", color=WHITE, fontsize=9)
                ax_lay.set_xticks([]); ax_lay.set_yticks([])
                ax_lay.set_xlim(0, board.width)
                ax_lay.set_ylim(0, board.height)
                ax_lay.set_aspect("equal")
    except Exception as e:
        ax_err = fig.add_subplot(gs[:, 2:])
        ax_err.set_facecolor(PANEL)
        ax_err.text(0.5, 0.5, f"Board render error:\n{e}",
                     transform=ax_err.transAxes, color=WHITE,
                     ha="center", va="center", fontsize=9)

    clear_output(wait=True)
    ipydisplay.display(fig)
    plt.close(fig)


print(f"Starting training for {CONFIG['TOTAL_TIMESTEPS']:,} timesteps...")
print(f"Checkpoints -> {CONFIG['CHECKPOINT_DIR']}")
print("-" * 60)

try:
    trainer.train(
        total_timesteps=CONFIG["TOTAL_TIMESTEPS"],
        on_update=on_update
    )
    print("Training complete!")
except KeyboardInterrupt:
    print("\nInterrupted — saving emergency checkpoint...")
    ep = os.path.join(CONFIG["CHECKPOINT_DIR"],
                      f"emergency_{trainer.total_timesteps}.pt")
    trainer.save_checkpoint(ep)
    print(f"Saved to: {ep}")


In [ ]:
# ================================================================
#  CELL 7 — LOAD CHECKPOINT + EVALUATE
# ================================================================
import os, sys, copy
import matplotlib.pyplot as plt
import IPython.display as ipydisplay
import torch, numpy as np

sys.path.insert(0, CONFIG["REPO_DIR"])
os.chdir(CONFIG["REPO_DIR"])

ckpt_dir = CONFIG["CHECKPOINT_DIR"]

if os.path.exists(ckpt_dir):
    ckpts = sorted([f for f in os.listdir(ckpt_dir) if f.endswith(".pt")])
    print(f"Checkpoints in {ckpt_dir}:")
    for c in ckpts:
        sz = os.path.getsize(os.path.join(ckpt_dir, c)) / 1024**2
        print(f"  {c}  ({sz:.1f} MB)")
else:
    ckpts = []
    print("No checkpoints found yet.")

EVAL_CHECKPOINT = CONFIG["LOAD_CHECKPOINT"]
# EVAL_CHECKPOINT = "/content/drive/MyDrive/pcb_router/checkpoints/checkpoint_50000.pt"
if not EVAL_CHECKPOINT and ckpts:
    EVAL_CHECKPOINT = os.path.join(ckpt_dir, ckpts[-1])
    print(f"\nAuto-selected: {ckpts[-1]}")

if not (EVAL_CHECKPOINT and os.path.exists(EVAL_CHECKPOINT)):
    print("No checkpoint found — train first (Cell 6) or set EVAL_CHECKPOINT manually.")
else:
    from pcb_router.training.trainer import PPOJEPATrainer, DreamerJEPATrainer, DreamerJEPATrainer
    from pcb_router.visualization.heatmap_viz import HeatmapVisualizer
    from pcb_router.visualization.renderer import BoardRenderer

    # Load using DreamerJEPATrainer or fallback
    try:
        eval_trainer = DreamerJEPATrainer(
            checkpoint_dir=ckpt_dir,
            load_checkpoint_path=EVAL_CHECKPOINT
        )
        is_dreamer = True
    except Exception as e:
        print(f"Dreamer load failed, falling back to PPO: {e}")
        eval_trainer = PPOJEPATrainer(
            checkpoint_dir=ckpt_dir,
            load_checkpoint_path=EVAL_CHECKPOINT
        )
        is_dreamer = False
    viz      = HeatmapVisualizer(theme_dark=True)
    renderer = BoardRenderer(theme_dark=True)

    obs, info = eval_trainer.env.reset()
    done, step, total_reward = False, 0, 0.0
    route_figs = []

    if is_dreamer:
        h, z = eval_trainer.jepa.initial_state(batch_size=1, device=eval_trainer.device)
        num_nets = len(eval_trainer.env.board.nets)
        
    print("\nRunning evaluation episode...")
    while not done and step < 200:
        raster     = torch.tensor(obs["board_raster"], dtype=torch.float32).unsqueeze(0).to(eval_trainer.device)
        layer_mask = torch.tensor(obs["layer_mask"],   dtype=torch.float32).unsqueeze(0).to(eval_trainer.device)
        graph      = info["graph"]
        x_dict          = {k: v.to(eval_trainer.device) for k, v in graph.x_dict.items()}         if hasattr(graph, "x_dict")         else {}
        edge_index_dict = {k: v.to(eval_trainer.device) for k, v in graph.edge_index_dict.items()} if hasattr(graph, "edge_index_dict") else {}

        with torch.no_grad():
            if is_dreamer:
                context_emb = eval_trainer.jepa.get_context_embedding(raster, x_dict, edge_index_dict, use_target=False)
                net_embs, umask, fs = eval_trainer._get_net_embeddings_and_mask(raster, x_dict, edge_index_dict)
                
                net_idx, heatmap_latent, _, _ = eval_trainer.policy.act(net_embs, umask, h, z, explore=False)
                
                action_emb = eval_trainer.jepa.get_action_embedding(net_idx, heatmap_latent)
                h, z, _, _ = eval_trainer.jepa.rssm_step(h, z, context_emb, action_emb)
                
                hv = eval_trainer.decoder(heatmap_latent, fs, eval_trainer.env.H, eval_trainer.env.W, active_layers_mask=layer_mask)
            else:
                sp, cls = eval_trainer.vit(raster)
                ne       = eval_trainer.gnn(x_dict, edge_index_dict)
                fp, fs   = eval_trainer.fusion(ne["pad"].unsqueeze(0), sp)
                num_nets  = len(eval_trainer.env.board.nets)
                net_embs  = torch.zeros((1, 100, eval_trainer.vit.embed_dim), device=eval_trainer.device)
                umask     = torch.zeros((1, 100), dtype=torch.bool, device=eval_trainer.device)
                for ni, net in enumerate(eval_trainer.env.board.nets):
                    pidx = [pi for pi, p in enumerate(eval_trainer.env.board.pins.values()) if p.net_id == net.id]
                    if pidx: net_embs[0, ni] = fp[0, pidx].mean(0)
                    if net.id not in eval_trainer.env.routed_nets: umask[0, ni] = True
                net_idx, hlat, _, _, _ = eval_trainer.policy(net_embs, umask, fs, cls, deterministic=True)
                hv = eval_trainer.decoder(hlat, fs, eval_trainer.env.H, eval_trainer.env.W, active_layers_mask=layer_mask)

        hmaps        = hv[0, :eval_trainer.env.board.num_layers].cpu().numpy()
        via_p        = hv[0, 8].cpu().numpy()
        board_before = copy.deepcopy(eval_trainer.env.board_state)

        next_obs, reward, terminated, truncated, next_info = eval_trainer.env.step_with_heatmaps(
            net_idx.item(), hmaps, via_p)
        total_reward += reward
        done = terminated or truncated

        if step % 5 == 0:
            try:
                snet = eval_trainer.env.board.nets[net_idx.item()] if net_idx.item() < num_nets else None
                spin = eval_trainer.env.board.pins[snet.pin_ids[0]] if snet else None
                hidx = spin.layer if (spin and spin.layer < len(hmaps)) else 0
                fig  = viz.render_routing_comparison(
                    board_before=board_before,
                    board_after=copy.deepcopy(eval_trainer.env.board_state),
                    heatmap=hmaps[hidx],
                    path=next_info.get("path", [])
                )
                route_figs.append(fig)
            except Exception: pass

        obs, info = next_obs, next_info
        step += 1
        print(f"  Step {step:3d} | reward: {reward:+.3f} | routed: {len(eval_trainer.env.routed_nets)}/{num_nets}", end="\r")

    cr  = next_info.get("completion_rate", 0)
    drc = next_info.get("drc_violations", 0)
    print(f"\n\nEvaluation done! Steps: {step} | Reward: {total_reward:.3f} | Completion: {cr:.1%} | DRC: {drc}")

    final_fig = renderer.render_board(eval_trainer.env.board_state, eval_trainer.env.board, show_all_layers=True)
    final_fig.suptitle(f"Final Routed Board — {cr:.1%} Completion", color="white", fontsize=14)
    ipydisplay.display(final_fig); plt.close(final_fig)

    print(f"\nStep-by-step comparisons ({len(route_figs)} frames):")
    for fig in route_figs:
        ipydisplay.display(fig); plt.close(fig)


In [ ]:
# ================================================================
#  CELL 8 — CHECKPOINT MANAGEMENT
# ================================================================
import os, torch

ckpt_dir = CONFIG["CHECKPOINT_DIR"]
print(f"Checkpoint directory: {ckpt_dir}\n")

if os.path.exists(ckpt_dir):
    ckpts = sorted([f for f in os.listdir(ckpt_dir) if f.endswith(".pt")])
    total_mb = 0
    for c in ckpts:
        full = os.path.join(ckpt_dir, c)
        sz   = os.path.getsize(full) / 1024**2
        total_mb += sz
        try:
            m  = torch.load(full, map_location="cpu", weights_only=False)
            ts = f"{m.get('total_timesteps', '?'):,}"
        except Exception:
            ts = "?"
        print(f"  {c:50s}  {sz:6.1f} MB  (step {ts})")
    print(f"\nTotal: {len(ckpts)} checkpoint(s), {total_mb:.1f} MB")
else:
    ckpts = []
    print("No checkpoint directory found — train first.")

# ---- Download options ----

DOWNLOAD_FILE        = None    # e.g. "checkpoint_50000.pt"
DOWNLOAD_ALL         = False   # zip + download all
COPY_LATEST_TO_DRIVE = False   # copy latest .pt to Drive root

if DOWNLOAD_FILE:
    from google.colab import files
    fp = os.path.join(ckpt_dir, DOWNLOAD_FILE)
    files.download(fp) if os.path.exists(fp) else print(f"Not found: {fp}")

if DOWNLOAD_ALL and ckpts:
    import shutil
    from google.colab import files
    shutil.make_archive("/content/pcb_checkpoints", "zip", ckpt_dir)
    files.download("/content/pcb_checkpoints.zip")
    print("Downloading all as zip...")

if COPY_LATEST_TO_DRIVE and ckpts:
    import shutil
    src = os.path.join(ckpt_dir, ckpts[-1])
    dst = f"/content/drive/MyDrive/{ckpts[-1]}"
    shutil.copy2(src, dst)
    print(f"Copied {ckpts[-1]} -> {dst}")
